# A4 — Colab driver

"Runtime → Run all". One linear path. Expected total on H100: ~90 min.

Pre-flight:
- Runtime → Change runtime type → **H100** GPU.
- Accept licences at https://huggingface.co/google/codegemma-7b-it and https://huggingface.co/google/gemma-3-1b-it.
- Have a GitHub classic PAT (`repo` scope) and an HF classic Read token handy.


## 1. Clone repo


In [ ]:
import os, subprocess, getpass

OWNER = 'D-Tony2020'
REPO  = 'LM-A3'
PROJECT_DIR = f'/content/{REPO}'

gh_token = getpass.getpass('GitHub PAT (repo scope): ')
if not os.path.exists(PROJECT_DIR):
    url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'
    subprocess.run(['git', 'clone', url, PROJECT_DIR], check=True)
    del url

os.chdir(PROJECT_DIR)
!git pull --quiet

# Keep the token in the remote URL so auto-submit can push back.
# Clear the local variable so it does not linger in notebook state.
del gh_token
!git log --oneline -3


## 2. Install dependencies


In [ ]:
!pip install -q transformers==4.51.3 accelerate==0.29.3 bitsandbytes==0.43.1 \
    sentencepiece==0.2.0 tokenizers==0.21.0 nltk==3.8.1 wandb==0.15.10 tqdm==4.66.1
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 3. HuggingFace login


In [ ]:
from huggingface_hub import login
import getpass
hf_tok = getpass.getpass('HF token (classic Read): ')
login(token=hf_tok, add_to_git_credential=False)
del hf_tok
print('HF logged in OK')


## 4. Cache dev ground-truth records (~40 s)


In [ ]:
!python cache_gt_records.py --split dev


## 5. Run all three tracks

`--auto-submit` commits and pushes after EACH successful track,
so a later failure never loses earlier results.

Order: T5 finetune (biggest scoring slot) → T5 from scratch → CodeGemma-7B LLM.


In [ ]:
!git config user.email "colab@runner"
!git config user.name  "A4 Colab Runner"
!python colab_train.py --task batch --auto-submit --config \
    "t5:t5_ft_h100,t5:t5_scr_h100,prompting:codegemma7b_k3_bm25_schema"


## 6. Dashboard


In [ ]:
!python colab_train.py --task dashboard


## 7. Final make_submission + push (catches anything auto-submit missed)


In [ ]:
!python make_submission.py
!git add results/ records/ experiments/registry.csv experiments/runs/
!git commit -m "Final: Colab session wrap-up" || echo "(nothing to commit)"
!git push origin main
